# Phase 20 — Overall Impression Signal Implementation

This notebook implements deterministic, evidence-backed `overallImpression` core signals. The model core emits a score, safe summary, evidence keys, and confidence notes only. Backend wrapper copy fields such as `topActionables` and `sectionReviews` stay outside model-core ownership.

## Step 20.1 — Deterministic evidence ledger

### Purpose
Build a deterministic evidence ledger from job-fit, ATS, matched and missing skills, role, experience, language, and parser confidence.

### Required input
Phase 18 job-fit examples, Phase 19 ATS scorer/fallback examples, Phase 19.5 clean skill signal audit, OpenAPI CV analyzer contract, and representative fixture rows.

### Action
Normalize raw model signals into evidence keys with explicit source fields and confidence flags. Remove unsupported skill/prose values before later summary generation.

### Expected output
`evidence_ledger` rows with stable keys, observed values, source fields, and confidence notes.

### Verification
Every ledger value must have a non-empty source field and every matched/missing skill must pass the safe skill filter.

In [66]:
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if not (ROOT / 'GAP_MODEL_TRAINING.md').exists():
    ROOT = Path.cwd().parent.parent
REPORTS = ROOT / 'reports'
NOTEBOOKS = ROOT / 'training' / 'notebooks'
REPORTS.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_20_overall_impression_signals'
SCHEMA_VERSION = 'overall-impression-signals-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()


def read_json(path: Path, default: Any) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text())

phase18_examples = read_json(REPORTS / 'phase_18_model_output_contract_examples.json', {}).get('examples', [])
phase19_fallbacks = read_json(REPORTS / 'phase_19_issue_errors_fallbacks.json', {}).get('fallback_examples', [])
phase195_clean = read_json(REPORTS / 'phase_19_5_clean_skill_signal_examples.json', {}).get('cleaned_examples', [])
openapi = read_json(ROOT / 'references/docs/generated/openapi.json', {})

SAFE_SKILL_RE = re.compile(r'^[a-z0-9][a-z0-9+#./ -]{1,48}[a-z0-9+#]$', re.I)
FORBIDDEN_SKILL_PHRASES = {
    'remote', 'competitive', 'opportunities', 'dynamic team', 'minimum', 'maximum',
    'years of experience', 'join our', 'large-scale ai workflows', 'growth'
}


def clean_skill(value: Any) -> str | None:
    text = str(value or '').strip().lower()
    if not text or len(text) > 50:
        return None
    if any(phrase in text for phrase in FORBIDDEN_SKILL_PHRASES):
        return None
    if not SAFE_SKILL_RE.match(text):
        return None
    return re.sub(r'\s+', ' ', text)

cleaned_by_pair = {row.get('pairId'): row for row in phase195_clean}

def score_band(score: int | float | None) -> str:
    if score is None:
        return 'unknown'
    if score >= 80:
        return 'high'
    if score >= 55:
        return 'medium'
    return 'low'

fixtures = [
    {
        'case_id': 'P20-en-high-fit-01',
        'language': 'EN',
        'target_role': 'Data Scientist',
        'job_fit': phase18_examples[1] if len(phase18_examples) > 1 else {'score': 82, 'matchedSkills': ['python', 'sql'], 'missingSkills': ['data mining']},
        'ats': {'score': 82, 'detectedIssues': ['formatting_risk_issue'], 'fallback': False, 'evidence': {'parserConfidence': 0.92, 'sectionCount': 5}},
        'experience': {'match': 'matched', 'evidence_key': 'experience_match'},
        'parser': {'confidence': 0.92, 'empty_parse': False},
    },
    {
        'case_id': 'P20-id-medium-fit-01',
        'language': 'ID',
        'target_role': 'Backend Developer',
        'job_fit': phase18_examples[0] if phase18_examples else {'score': 73, 'matchedSkills': ['python', 'apis'], 'missingSkills': []},
        'ats': {'score': 100, 'detectedIssues': [], 'fallback': False, 'evidence': {'parserConfidence': 0.90, 'sectionCount': 4}},
        'experience': {'match': 'matched', 'evidence_key': 'experience_match'},
        'parser': {'confidence': 0.90, 'empty_parse': False},
    },
    {
        'case_id': 'P20-empty-parse-01',
        'language': 'ID',
        'target_role': 'Backend Developer',
        'job_fit': {'score': None, 'matchedSkills': [], 'missingSkills': []},
        'ats': phase19_fallbacks[0]['atsFriendliness'] if phase19_fallbacks else {'score': 35, 'detectedIssues': ['empty_parse_risk', 'parseability_issue'], 'fallback': True},
        'experience': {'match': 'unknown', 'evidence_key': 'parser_empty'},
        'parser': {'confidence': 0.0, 'empty_parse': True},
    },
    {
        'case_id': 'P20-missing-job-01',
        'language': 'EN',
        'target_role': None,
        'job_fit': {'score': None, 'matchedSkills': ['python'], 'missingSkills': []},
        'ats': {'score': 84, 'detectedIssues': [], 'fallback': False},
        'experience': {'match': 'unknown', 'evidence_key': 'missing_target_job'},
        'parser': {'confidence': 0.88, 'empty_parse': False},
    },
    {
        'case_id': 'P20-unknown-language-01',
        'language': 'UNKNOWN',
        'target_role': 'Security Analyst',
        'job_fit': phase18_examples[2] if len(phase18_examples) > 2 else {'score': 5, 'matchedSkills': [], 'missingSkills': ['incident response']},
        'ats': {'score': 70, 'detectedIssues': ['metric_evidence_issue'], 'fallback': False},
        'experience': {'match': 'unknown', 'evidence_key': 'unknown_language'},
        'parser': {'confidence': 0.72, 'empty_parse': False},
    },
    {
        'case_id': 'P20-low-confidence-sparse-01',
        'language': 'EN',
        'target_role': 'Software Engineer',
        'job_fit': {'score': 42, 'matchedSkills': [], 'missingSkills': ['testing']},
        'ats': {'score': 58, 'detectedIssues': ['section_completeness_issue'], 'fallback': False},
        'experience': {'match': 'unknown', 'evidence_key': 'sparse_evidence'},
        'parser': {'confidence': 0.45, 'empty_parse': False},
    },
]


def build_evidence_ledger(row: dict[str, Any]) -> dict[str, Any]:
    job_fit = row['job_fit']
    pair_id = job_fit.get('pairId')
    cleaned = cleaned_by_pair.get(pair_id, {})
    matched = [s for s in (clean_skill(v) for v in cleaned.get('matchedSkills', job_fit.get('matchedSkills', []))) if s]
    missing = [s for s in (clean_skill(v) for v in cleaned.get('missingSkills', job_fit.get('missingSkills', []))) if s]
    parser_conf = float(row.get('parser', {}).get('confidence', row.get('ats', {}).get('evidence', {}).get('parserConfidence', 0.0)) or 0.0)
    job_score = job_fit.get('score')
    ats_score = row['ats'].get('score')
    entries = [
        {'key': 'target_role', 'value': row.get('target_role'), 'source': 'target_job.role'},
        {'key': 'language', 'value': row.get('language'), 'source': 'language_normalization.output'},
        {'key': 'jobfit_score', 'value': job_score, 'source': 'jobFitAlignment.score'},
        {'key': 'jobfit_band', 'value': score_band(job_score), 'source': 'jobFitAlignment.score'},
        {'key': 'matched_skills', 'value': matched, 'source': 'jobFitAlignment.matchedSkills'},
        {'key': 'missing_skills', 'value': missing[:3], 'source': 'jobFitAlignment.missingSkills'},
        {'key': 'ats_score', 'value': ats_score, 'source': 'atsFriendliness.score'},
        {'key': 'ats_issues', 'value': row['ats'].get('detectedIssues', []), 'source': 'atsFriendliness.detectedIssues'},
        {'key': 'experience_match', 'value': row['experience'].get('match'), 'source': row['experience'].get('evidence_key')},
        {'key': 'parser_confidence', 'value': parser_conf, 'source': 'parser.confidence'},
        {'key': 'empty_parse', 'value': bool(row.get('parser', {}).get('empty_parse') or row['ats'].get('fallback')), 'source': 'parser.empty_parse|atsFriendliness.fallback'},
    ]
    confidence_notes = []
    if parser_conf < 0.50:
        confidence_notes.append('low_parser_confidence')
    if row.get('language') not in {'ID', 'EN'}:
        confidence_notes.append('unsupported_or_unknown_language')
    if not row.get('target_role'):
        confidence_notes.append('missing_target_job')
    if not matched and not missing:
        confidence_notes.append('sparse_skill_evidence')
    if row['ats'].get('fallback'):
        confidence_notes.append('ats_safe_fallback_used')
    if score_band(job_score) == 'low':
        confidence_notes.append('low_jobfit_alignment')
    return {'case_id': row['case_id'], 'entries': entries, 'confidence_notes': confidence_notes}

evidence_ledger = [build_evidence_ledger(row) for row in fixtures]

ledger_errors = []
for ledger in evidence_ledger:
    for entry in ledger['entries']:
        if not entry.get('source'):
            ledger_errors.append({'case_id': ledger['case_id'], 'key': entry['key'], 'error': 'missing_source'})
    skills = next(e['value'] for e in ledger['entries'] if e['key'] == 'matched_skills') + next(e['value'] for e in ledger['entries'] if e['key'] == 'missing_skills')
    for skill in skills:
        if clean_skill(skill) != skill:
            ledger_errors.append({'case_id': ledger['case_id'], 'skill': skill, 'error': 'unsafe_skill'})

assert not ledger_errors, ledger_errors
len(evidence_ledger), evidence_ledger[0]['case_id']

(6, 'P20-en-high-fit-01')

## Step 20.2 — Safe ID and EN summary templates

### Purpose
Generate safe summaries in Indonesian and English that mention only observed evidence.

### Required input
Evidence ledger entries for job-fit band, matched skills, missing skills, ATS issues, target role, language, and confidence notes.

### Action
Apply deterministic templates by language. Fill slots only from evidence keys and keep fallback/template selection explicit.

### Expected output
Representative `overallImpression.summary` strings for `ID` and `EN`.

### Verification
Each generated summary must carry `used_evidence_keys`; no template can add skill, seniority, outcome, or product-copy claims without evidence.

In [67]:
TEMPLATES = {
    'EN': {
        'high': 'CV shows strong alignment for {role} based on {matched}; review {missing}{ats_clause}.',
        'medium': 'CV shows partial alignment for {role} with evidence in {matched}; strengthen {missing}{ats_clause}.',
        'low': 'CV has limited observed alignment for {role}; prioritize evidence for {missing}{ats_clause}.',
        'fallback': 'CV can be reviewed only with limited evidence; {reason}.',
    },
    'ID': {
        'high': 'CV menunjukkan keselarasan kuat untuk {role} berdasarkan {matched}; tinjau {missing}{ats_clause}.',
        'medium': 'CV menunjukkan keselarasan sebagian untuk {role} dengan bukti pada {matched}; perkuat {missing}{ats_clause}.',
        'low': 'CV memiliki bukti keselarasan terbatas untuk {role}; prioritaskan bukti untuk {missing}{ats_clause}.',
        'fallback': 'CV hanya bisa ditinjau dengan bukti terbatas; {reason}.',
    },
}

ATS_LABELS = {
    'EN': {
        'formatting_risk_issue': 'ATS formatting risk',
        'metric_evidence_issue': 'limited measurable impact evidence',
        'section_completeness_issue': 'incomplete section evidence',
        'parseability_issue': 'parseability risk',
        'empty_parse_risk': 'empty CV parse risk',
    },
    'ID': {
        'formatting_risk_issue': 'risiko format ATS',
        'metric_evidence_issue': 'bukti dampak terukur masih terbatas',
        'section_completeness_issue': 'bukti section belum lengkap',
        'parseability_issue': 'risiko keterbacaan parser',
        'empty_parse_risk': 'risiko hasil parse CV kosong',
    },
}


def ledger_value(ledger: dict[str, Any], key: str) -> Any:
    for entry in ledger['entries']:
        if entry['key'] == key:
            return entry['value']
    return None


def join_values(values: list[str], language: str, empty_label: str) -> str:
    values = [v for v in values if v]
    if not values:
        return empty_label
    if len(values) == 1:
        return values[0]
    sep = ', '
    return sep.join(values[:3])


def build_summary(ledger: dict[str, Any]) -> dict[str, Any]:
    language = ledger_value(ledger, 'language')
    lang = language if language in {'ID', 'EN'} else 'EN'
    role = ledger_value(ledger, 'target_role')
    matched = ledger_value(ledger, 'matched_skills') or []
    missing = ledger_value(ledger, 'missing_skills') or []
    ats_issues = ledger_value(ledger, 'ats_issues') or []
    band = ledger_value(ledger, 'jobfit_band')
    notes = list(ledger['confidence_notes'])
    used = ['language', 'target_role', 'jobfit_band', 'matched_skills', 'missing_skills', 'ats_issues', 'parser_confidence']

    if ledger_value(ledger, 'empty_parse'):
        reason = 'hasil parse CV kosong atau tidak cukup terbaca' if lang == 'ID' else 'CV parse is empty or not reliable enough'
        return {'summary': TEMPLATES[lang]['fallback'].format(reason=reason), 'template_id': f'{lang.lower()}_empty_parse_fallback_v1', 'used_evidence_keys': used, 'confidence_notes': notes}
    if not role:
        reason = 'target pekerjaan belum tersedia' if lang == 'ID' else 'target job is missing'
        return {'summary': TEMPLATES[lang]['fallback'].format(reason=reason), 'template_id': f'{lang.lower()}_missing_target_job_fallback_v1', 'used_evidence_keys': used, 'confidence_notes': notes}
    if language not in {'ID', 'EN'}:
        reason = 'language is unknown, so English safe fallback is used'
        return {'summary': TEMPLATES['EN']['fallback'].format(reason=reason), 'template_id': 'en_unknown_language_fallback_v1', 'used_evidence_keys': used, 'confidence_notes': notes}
    if 'low_parser_confidence' in notes or (not matched and not missing):
        reason = 'bukti masih jarang atau keyakinan parser rendah' if lang == 'ID' else 'evidence is sparse or parser confidence is low'
        return {'summary': TEMPLATES[lang]['fallback'].format(reason=reason), 'template_id': f'{lang.lower()}_low_confidence_fallback_v1', 'used_evidence_keys': used, 'confidence_notes': notes}

    matched_text = join_values(matched, lang, 'observed role evidence' if lang == 'EN' else 'bukti role yang teramati')
    missing_text = join_values(missing, lang, 'remaining requirement evidence' if lang == 'EN' else 'bukti kebutuhan yang tersisa')
    issue_labels = [ATS_LABELS[lang].get(issue, issue) for issue in ats_issues]
    ats_clause = ''
    if issue_labels:
        ats_clause = (' and note ' if lang == 'EN' else ' dan perhatikan ') + join_values(issue_labels, lang, issue_labels[0])
    summary = TEMPLATES[lang].get(band, TEMPLATES[lang]['medium']).format(role=role, matched=matched_text, missing=missing_text, ats_clause=ats_clause)
    return {'summary': summary, 'template_id': f'{lang.lower()}_{band}_evidence_template_v1', 'used_evidence_keys': used, 'confidence_notes': notes}

summary_examples = [{**{'case_id': ledger['case_id']}, **build_summary(ledger)} for ledger in evidence_ledger]
assert any(row['template_id'].startswith('id_') for row in summary_examples)
assert any(row['template_id'].startswith('en_') for row in summary_examples)
summary_examples[:2]

[{'case_id': 'P20-en-high-fit-01',
  'summary': 'CV shows strong alignment for Data Scientist based on deep learning, machine learning, python; review artificial intelligence, computer vision, data engineering and note ATS formatting risk.',
  'template_id': 'en_high_evidence_template_v1',
  'used_evidence_keys': ['language',
   'target_role',
   'jobfit_band',
   'matched_skills',
   'missing_skills',
   'ats_issues',
   'parser_confidence'],
  'confidence_notes': []},
 {'case_id': 'P20-id-medium-fit-01',
  'summary': 'CV menunjukkan keselarasan sebagian untuk Backend Developer dengan bukti pada apis, python; perkuat bukti kebutuhan yang tersisa.',
  'template_id': 'id_medium_evidence_template_v1',
  'used_evidence_keys': ['language',
   'target_role',
   'jobfit_band',
   'matched_skills',
   'missing_skills',
   'ats_issues',
   'parser_confidence'],
  'confidence_notes': []}]

## Step 20.3 — Safe fallback summaries

### Purpose
Return controlled summaries for empty CV parse, missing target job, unknown language, low confidence, and sparse evidence.

### Required input
Evidence ledger confidence notes and parser/job/language fields.

### Action
Select fallback template IDs before normal templates when evidence is insufficient or unsafe.

### Expected output
Fallback examples with explicit reason and evidence keys.

### Verification
Required fallback case IDs must use fallback template IDs and must not emit unsupported skill, seniority, or hiring-outcome claims.

In [68]:
required_fallback_cases = {
    'P20-empty-parse-01': 'empty_parse_fallback',
    'P20-missing-job-01': 'missing_target_job_fallback',
    'P20-unknown-language-01': 'unknown_language_fallback',
    'P20-low-confidence-sparse-01': 'low_confidence_fallback',
}

fallback_checks = []
for row in summary_examples:
    expected = required_fallback_cases.get(row['case_id'])
    if expected and expected not in row['template_id']:
        fallback_checks.append({'case_id': row['case_id'], 'expected': expected, 'actual': row['template_id']})

assert not fallback_checks, fallback_checks
fallback_examples = [row for row in summary_examples if 'fallback' in row['template_id']]
fallback_examples

[{'case_id': 'P20-empty-parse-01',
  'summary': 'CV hanya bisa ditinjau dengan bukti terbatas; hasil parse CV kosong atau tidak cukup terbaca.',
  'template_id': 'id_empty_parse_fallback_v1',
  'used_evidence_keys': ['language',
   'target_role',
   'jobfit_band',
   'matched_skills',
   'missing_skills',
   'ats_issues',
   'parser_confidence'],
  'confidence_notes': ['low_parser_confidence',
   'sparse_skill_evidence',
   'ats_safe_fallback_used']},
 {'case_id': 'P20-missing-job-01',
  'summary': 'CV can be reviewed only with limited evidence; target job is missing.',
  'template_id': 'en_missing_target_job_fallback_v1',
  'used_evidence_keys': ['language',
   'target_role',
   'jobfit_band',
   'matched_skills',
   'missing_skills',
   'ats_issues',
   'parser_confidence'],
  'confidence_notes': ['missing_target_job']},
 {'case_id': 'P20-unknown-language-01',
  'summary': 'CV can be reviewed only with limited evidence; language is unknown, so English safe fallback is used.',
  'temp

## Step 20.4 — Hallucination and claim validation

### Purpose
Reject hallucinated skills, unsupported seniority, hiring-outcome claims, and language mismatch before export.

### Required input
Generated summaries, evidence keys, allowed skill values, selected language, and forbidden claim policy.

### Action
Validate summary text against evidence vocabulary and explicit forbidden patterns.

### Expected output
A validation report with zero errors for generated examples and positive rejection checks for deliberately invalid candidates.

### Verification
Generated examples pass. Deliberate invalid candidates are rejected for the expected reasons.

In [69]:
FORBIDDEN_OUTCOME_RE = re.compile(r'\b(will be hired|guaranteed|interview|diterima|pasti lolos|peluang diterima)\b', re.I)
UNSUPPORTED_SENIORITY_RE = re.compile(r'\b(senior|lead|manager|junior-mid|junior|mid-level)\b', re.I)
INDONESIAN_MARKERS = {'cv', 'menunjukkan', 'keselarasan', 'bukti', 'perkuat', 'tinjau', 'terbatas', 'pekerjaan'}
ENGLISH_MARKERS = {'cv', 'shows', 'alignment', 'evidence', 'review', 'strengthen', 'limited', 'target'}


def validate_summary(row: dict[str, Any], ledger: dict[str, Any]) -> list[dict[str, str]]:
    errors: list[dict[str, str]] = []
    text = row['summary']
    lang = ledger_value(ledger, 'language')
    matched = set(ledger_value(ledger, 'matched_skills') or [])
    missing = set(ledger_value(ledger, 'missing_skills') or [])
    allowed_skills = matched | missing
    lower = text.lower()

    if FORBIDDEN_OUTCOME_RE.search(text):
        errors.append({'reason': 'hiring_outcome_claim'})
    if UNSUPPORTED_SENIORITY_RE.search(text):
        errors.append({'reason': 'unsupported_seniority'})

    for skill in re.findall(r'\b(?:kubernetes|golang|rust|react|python|sql|testing|data mining|machine learning|deep learning|apis)\b', lower):
        if skill not in allowed_skills:
            errors.append({'reason': 'hallucinated_skill', 'value': skill})

    words = set(re.findall(r'[a-zA-Z]+', lower))
    if lang == 'ID' and len(words & ENGLISH_MARKERS) > len(words & INDONESIAN_MARKERS) + 2:
        errors.append({'reason': 'language_mismatch'})
    if lang == 'EN' and len(words & INDONESIAN_MARKERS) > len(words & ENGLISH_MARKERS) + 2:
        errors.append({'reason': 'language_mismatch'})
    if not row.get('used_evidence_keys'):
        errors.append({'reason': 'missing_evidence_keys'})
    return errors

ledger_by_case = {ledger['case_id']: ledger for ledger in evidence_ledger}
validation_results = []
for row in summary_examples:
    errors = validate_summary(row, ledger_by_case[row['case_id']])
    validation_results.append({'case_id': row['case_id'], 'passed': not errors, 'errors': errors})

negative_candidates = [
    {'case_id': 'P20-en-high-fit-01', 'summary': 'CV shows strong Kubernetes alignment and will be hired.', 'used_evidence_keys': ['matched_skills']},
    {'case_id': 'P20-id-medium-fit-01', 'summary': 'CV shows strong Python alignment for senior roles.', 'used_evidence_keys': ['matched_skills']},
]
negative_rejections = []
for row in negative_candidates:
    errors = validate_summary(row, ledger_by_case[row['case_id']])
    negative_rejections.append({'case_id': row['case_id'], 'errors': errors})

assert all(item['passed'] for item in validation_results), validation_results
assert any(e['reason'] == 'hallucinated_skill' for item in negative_rejections for e in item['errors'])
assert any(e['reason'] == 'hiring_outcome_claim' for item in negative_rejections for e in item['errors'])
assert any(e['reason'] == 'unsupported_seniority' for item in negative_rejections for e in item['errors'])
validation_results, negative_rejections

([{'case_id': 'P20-en-high-fit-01', 'passed': True, 'errors': []},
  {'case_id': 'P20-id-medium-fit-01', 'passed': True, 'errors': []},
  {'case_id': 'P20-empty-parse-01', 'passed': True, 'errors': []},
  {'case_id': 'P20-missing-job-01', 'passed': True, 'errors': []},
  {'case_id': 'P20-unknown-language-01', 'passed': True, 'errors': []},
  {'case_id': 'P20-low-confidence-sparse-01', 'passed': True, 'errors': []}],
 [{'case_id': 'P20-en-high-fit-01',
   'errors': [{'reason': 'hiring_outcome_claim'},
    {'reason': 'hallucinated_skill', 'value': 'kubernetes'}]},
  {'case_id': 'P20-id-medium-fit-01',
   'errors': [{'reason': 'unsupported_seniority'}]}])

## Step 20.5 — Export model-core overall impression contract

### Purpose
Export core `overallImpression.score`, `summary`, evidence keys, and confidence notes for backend wrapper rendering.

### Required input
Evidence ledger, summary examples, validation results, OpenAPI response boundary, and wrapper-owned field policy.

### Action
Compute transparent overall scores from job-fit, ATS, parser confidence, and confidence penalties. Export examples and validation metadata.

### Expected output
`reports/phase_20_overall_impression_signals.json` with schema version, examples, validation, forbidden wrapper fields, and acceptance status.

### Verification
Scores stay within `0-100`, summaries pass validation, every example has evidence keys, and wrapper-owned fields are absent from exported model-core examples.

In [70]:
WRAPPER_OWNED_FIELDS = {'topActionables', 'sectionReviews'}
MODEL_CORE_FIELDS = {'overallImpression'}


def clamp_score(value: float) -> int:
    return int(max(0, min(100, round(value))))


def overall_score(ledger: dict[str, Any]) -> int:
    job_score = ledger_value(ledger, 'jobfit_score')
    ats_score = ledger_value(ledger, 'ats_score')
    parser_conf = ledger_value(ledger, 'parser_confidence') or 0.0
    if job_score is None:
        base = (ats_score or 0) * 0.35
    else:
        base = float(job_score) * 0.65 + float(ats_score or 0) * 0.25 + float(parser_conf) * 100 * 0.10
    penalty = 0
    if 'low_parser_confidence' in ledger['confidence_notes']:
        penalty += 15
    if 'missing_target_job' in ledger['confidence_notes']:
        penalty += 20
    if 'unsupported_or_unknown_language' in ledger['confidence_notes']:
        penalty += 10
    if 'ats_safe_fallback_used' in ledger['confidence_notes']:
        penalty += 20
    return clamp_score(base - penalty)

exports = []
for row in summary_examples:
    ledger = ledger_by_case[row['case_id']]
    payload = {
        'caseId': row['case_id'],
        'overallImpression': {
            'score': overall_score(ledger),
            'summary': row['summary'],
            'evidenceKeys': row['used_evidence_keys'],
            'confidenceNotes': row['confidence_notes'],
            'templateId': row['template_id'],
        },
        'model': {'name': PHASE_ID, 'version': SCHEMA_VERSION},
    }
    exports.append(payload)

export_errors = []
for item in exports:
    oi = item['overallImpression']
    if not 0 <= oi['score'] <= 100:
        export_errors.append({'case_id': item['caseId'], 'reason': 'score_out_of_range'})
    if not oi['evidenceKeys']:
        export_errors.append({'case_id': item['caseId'], 'reason': 'missing_evidence_keys'})
    if WRAPPER_OWNED_FIELDS & set(item):
        export_errors.append({'case_id': item['caseId'], 'reason': 'wrapper_owned_field_exported'})
    if WRAPPER_OWNED_FIELDS & set(oi):
        export_errors.append({'case_id': item['caseId'], 'reason': 'wrapper_owned_field_exported'})

assert not export_errors, export_errors

acceptance = {
    'traceable_to_evidence_keys': all(item['overallImpression']['evidenceKeys'] for item in exports),
    'unsupported_claims_rejected': all(item['passed'] for item in validation_results) and bool(negative_rejections),
    'id_and_en_templates_tested': any(row['template_id'].startswith('id_') for row in summary_examples) and any(row['template_id'].startswith('en_') for row in summary_examples),
    'wrapper_owned_fields_excluded': all(not (WRAPPER_OWNED_FIELDS & set(item)) and not (WRAPPER_OWNED_FIELDS & set(item['overallImpression'])) for item in exports),
}

report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'source_reports': [
        'reports/phase_18_model_output_contract_examples.json',
        'reports/phase_19_ats_scorer_metrics.json',
        'reports/phase_19_issue_errors_fallbacks.json',
        'reports/phase_19_5_clean_skill_signal_examples.json',
    ],
    'contract_boundary': {
        'model_core_owns': ['overallImpression.score', 'overallImpression.summary', 'overallImpression.evidenceKeys', 'overallImpression.confidenceNotes'],
        'wrapper_owns': sorted(WRAPPER_OWNED_FIELDS),
        'public_contract_observed': bool(openapi.get('components', {}).get('schemas', {}).get('CvAnalysis')),
    },
    'evidence_ledger': evidence_ledger,
    'summary_examples': summary_examples,
    'fallback_examples': fallback_examples,
    'validation_results': validation_results,
    'negative_rejections': negative_rejections,
    'export_examples': exports,
    'acceptance': acceptance,
    'status': 'complete' if all(acceptance.values()) else 'blocked',
}

(REPORTS / 'phase_20_overall_impression_signals.json').write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
assert report['status'] == 'complete', acceptance
report['status'], len(exports)

('complete', 6)